In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# исследуемые модели
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

from sklearn.model_selection import RandomizedSearchCV
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import ParameterGrid

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
np.random.seed(42)

In [ ]:
train_df = pd.read_csv("purchases_data_20000.csv")

In [ ]:
train_df.head()

In [ ]:
print(train_df.info())

In [ ]:
print(train_df.isnull().sum())

In [ ]:
cat_train_df = train_df[["Название магазина", "Категория", "Бренд", "Номер карты"]]
num_train_df = train_df[["Дата и время", "Долгота", "Широта", "Количество товаров", "Стоимость"]]

In [ ]:
num_train_df['Дата и время'] = pd.to_datetime(num_train_df['Дата и время'])

num_train_df['Год'] = num_train_df['Дата и время'].dt.year
num_train_df['Месяц'] = num_train_df['Дата и время'].dt.month
num_train_df['День'] = num_train_df['Дата и время'].dt.day
# num_train_df['Час'] = num_train_df['Дата и время'].dt.hour
# num_train_df['Минута'] = num_train_df['Дата и время'].dt.minute

num_train_df = num_train_df.drop("Дата и время", axis=1)

In [ ]:
num_train_df.head()

In [ ]:
# num_train_cols = [col for col in num_train_df.columns if col not in ['Стоимость']]

# plt.figure(figsize=(15, len(num_train_cols) * 3))
# for i, col in enumerate(num_train_cols, 1):
#     plt.subplot(len(num_train_cols), 2, 2*i-1)
#     sns.histplot(num_train_df[col].dropna(), kde=True)  # kde добавляет линию плотности
#     plt.title(f'Histogram of {col}')
    
#     plt.subplot(len(num_train_cols), 2, 2*i)
#     sns.boxplot(x=num_train_df[col].dropna())
#     plt.title(f'Boxplot of {col}')
    

# plt.tight_layout()
# plt.show()

In [ ]:
# num_train_df_filled = num_train_df.fillna(num_train_df.median())

# corr_matrix = num_train_df_filled.corr(method='spearman')

# plt.figure(figsize=(10, 8))
# sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
# plt.title('Spearman Correlation Matrix')
# plt.show()

### Категориальные

In [ ]:
scaler = StandardScaler()
num_train_features = num_train_df.columns
num_train_df[num_train_features] = scaler.fit_transform(num_train_df[num_train_features])

In [ ]:
cat_train_df.head()

In [ ]:
investigated_column = "Стоимость"

In [ ]:
for column in cat_train_df.columns:
    unique_count = cat_train_df[column].nunique()
    print(f"{column}: {unique_count} уникальных категорий")

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# One-Hot Encoding
onehot_cols = ['Название магазина', 'Категория']
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')
onehot_encoded = onehot_encoder.fit_transform(cat_train_df[onehot_cols])
onehot_train_df = pd.DataFrame(onehot_encoded, columns=onehot_encoder.get_feature_names_out(onehot_cols))

# 2. Frequency Encoding
freq_cols = ['Бренд', 'Номер карты']
for col in freq_cols:
    freq_encoding = cat_train_df[col].value_counts(normalize=True)
    cat_train_df[col + '_freq'] = cat_train_df[col].map(freq_encoding)

cat_encoded_train_df = pd.concat([onehot_train_df, cat_train_df[[col + '_freq' for col in freq_cols]]], axis=1)

In [ ]:
numeric_cols = ['Долгота', 'Широта', 'Количество товаров', 'Стоимость', 'Год', 'Месяц', 'День']
numeric_cols.remove(investigated_column)
num_train_features = num_train_df[numeric_cols]

In [ ]:
X = pd.concat([num_train_features, cat_encoded_train_df], axis=1)

y = num_train_df[investigated_column]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
def print_top_models(grid_search, model_name):
    cv_results = grid_search.cv_results_
    
    results_df = pd.DataFrame({
        'params': cv_results['params'],
        'mean_f1': cv_results['mean_test_score'],
        'std_f1': cv_results['std_test_score']
    })
    
    top_models = results_df.sort_values(by='mean_f1', ascending=False).head(5)
    
    def format_params(params):
        formatted = {}
        for k, v in params.items():
            key = k.replace('model__', '')
            if isinstance(v, float):
                if abs(v) >= 1000 or abs(v) < 0.001:
                    formatted_val = f"{v:.2e}"
                else:
                    formatted_val = f"{v:.4f}".rstrip('0').rstrip('.')
                formatted[key] = formatted_val
            else:
                formatted[key] = str(v)
        return formatted
    
    print(f"\n{'='*50}")
    print(f"Top 5 {model_name} models by F1-score")
    print("="*50)
    
    for i, (index, row) in enumerate(top_models.iterrows(), 1):
        params = format_params(row['params'])
        print(f"\n#{i} | F1: {row['mean_f1']:.4f} ± {row['std_f1']:.4f}")
        print("─" * 50)
        
        line = []
        max_line_length = 80
        for param, value in params.items():
            item = f"{param}: {value}"
            if len(', '.join(line + [item])) > max_line_length and line:
                print(', '.join(line))
                line = []
            line.append(item)
        if line:
            print(', '.join(line))

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('model', Ridge(random_state=42)) 
])

param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}


grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=cv,
    n_jobs=-1, 
    verbose=1,
    error_score='raise'
)

print("Запуск поиска гиперпараметров с 10-кратной кросс-валидацией...")
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("\nМетрики на валидационной выборке:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")